# EDA: Датасет для детекции дублей в аналитических таксономиях

**ВКР:** Разработка инструмента для автоматической идентификации дублирующихся событий  
и проверки стандартов именования в аналитических таксономиях

## Содержание
1. [Источник данных и описание задачи](#sec1)
2. [Загрузка и первичный осмотр](#sec2)
3. [Предобработка и feature engineering](#sec3)
4. [EDA: структура датасета](#sec4)
5. [EDA: символьное сходство](#sec5)
6. [EDA: лексические признаки](#sec6)
7. [EDA: корреляции и форматы именования](#sec7)
8. [Train / Val / Test split](#sec8)
9. [Выводы](#sec9)

---
## 1. Источник данных и описание задачи <a id='sec1'></a>

### Задача
Бинарная классификация пар аналитических событий: **дубль** (`label=1`) или **не дубль** (`label=0`).

### Источник
Датасет сформирован на основе реальной таксономии Альфа-Банка методом контролируемой аугментации.  
Seed-данные — 4 027 уникальных событий из производственного трекинг-плана.

### Типология дублей

| Тип | Описание | Пример |
|-----|----------|--------|
| `exact` | Точное совпадение | `account_click` / `account_click` |
| `typo` | Опечатка (удаление / дублирование / замена символа) | `account_click` / `acccount_click` |
| `case` | Смена формата именования (8 вариантов) | `account_click` / `AccountClick` |
| `synonym` | Синоним action-глагола | `account_view` / `account_open` |
| `permutation` | Перестановка токенов объектного пути | `card_payment_click` / `payment_card_click` |
| `semantic` | Семантический дубль через замену токенов объекта | `account_screen_view` / `wallet_page_open` |
| `not_duplicate` | Не дубль (hard negatives) | `login_success` / `card_reissue_click` |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import difflib, re, warnings
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
np.random.seed(42)

plt.rcParams.update({
    'figure.dpi': 120, 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3,
})

PALETTE = {
    'exact': '#1abc9c', 'typo': '#3498db', 'case': '#9b59b6',
    'synonym': '#e67e22', 'permutation': '#e74c3c',
    'semantic': '#c0392b', 'not_duplicate': '#95a5a6',
}
TYPE_ORDER = ['exact','typo','case','synonym','permutation','semantic','not_duplicate']

print('Импорты выполнены.')

---
## 2. Загрузка и первичный осмотр <a id='sec2'></a>

In [ ]:
df = pd.read_csv('data/taxonomy_duplicates_balanced.csv')

print(f'Строк: {len(df):,}')
print(f'Столбцов: {df.shape[1]}')
print(f'NaN: {df.isnull().sum().sum()}')
print(f'Дублирующихся пар: {df.duplicated(subset=["event_1","event_2"]).sum()}')
print()
df.head(8)

In [ ]:
print('=== Распределение классов ===')
dist = df.groupby(['label','dup_type']).size().reset_index(name='n')
dist['%'] = (dist['n'] / len(df) * 100).round(2)
print(dist.to_string(index=False))
print(f'\nDuplicate rate: {df.label.mean():.3f} ({df.label.mean()*100:.1f}%)')
print(f'Уникальных событий: {pd.concat([df.event_1,df.event_2]).nunique():,}')

---
## 3. Предобработка и feature engineering <a id='sec3'></a>

In [ ]:
def to_snake(s):
    """Нормализует любой формат именования в snake_case."""
    s = str(s).strip()
    s = re.sub(r'[>/\\-]+', ' ', s)
    s = re.sub(r'([a-z])([A-Z])', r'\1 \2', s)
    s = re.sub(r'([A-Z]+)([A-Z][a-z])', r'\1 \2', s)
    s = re.sub(r'\s+', '_', s); s = s.lower()
    s = re.sub(r'[^a-z0-9_]', '_', s)
    s = re.sub(r'_+', '_', s).strip('_')
    return s

def char_sim(a, b):
    return round(difflib.SequenceMatcher(None, str(a), str(b)).ratio(), 4)

def token_jaccard(a, b):
    sa, sb = set(str(a).split('_')), set(str(b).split('_'))
    return len(sa & sb) / len(sa | sb) if (sa | sb) else 1.0

def detect_format(s):
    s = str(s)
    if '_' in s and s == s.lower():                            return 'snake_case'
    if re.match(r'^[A-Z_]+$', s):                             return 'SCREAMING_SNAKE'
    if re.match(r'^[A-Z][a-z]+([A-Z][a-z]+)+$', s):          return 'PascalCase'
    if re.match(r'^[a-z]+([A-Z][a-z]+)+$', s):               return 'camelCase'
    if '-' in s and s == s.lower():                           return 'kebab-case'
    if '-' in s:                                              return 'Train-Case'
    if s == s.lower() and '_' not in s and '-' not in s:      return 'flatcase'
    if s == s.upper() and '_' not in s:                       return 'UPPERCASE'
    return 'other'

# Вычисляем признаки
df['len_1']       = df['event_1'].apply(len)
df['len_2']       = df['event_2'].apply(len)
df['tok_1']       = df['event_1'].apply(lambda x: len(str(x).split('_')))
df['tok_2']       = df['event_2'].apply(lambda x: len(str(x).split('_')))
df['len_diff']    = (df['len_1'] - df['len_2']).abs()
df['tok_diff']    = (df['tok_1'] - df['tok_2']).abs()
df['char_sim']    = df.apply(lambda r: char_sim(r.event_1, r.event_2), axis=1)
df['jaccard']     = df.apply(lambda r: round(token_jaccard(r.event_1, r.event_2), 4), axis=1)
df['norm_1']      = df['event_1'].apply(to_snake)
df['norm_2']      = df['event_2'].apply(to_snake)
df['norm_sim']    = df.apply(lambda r: char_sim(r.norm_1, r.norm_2), axis=1)
df['fmt_1']       = df['event_1'].apply(detect_format)
df['same_action'] = (df['event_1'].apply(lambda x: str(x).split('_')[-1]) ==
                     df['event_2'].apply(lambda x: str(x).split('_')[-1])).astype(int)
df['same_domain'] = (df['event_1'].apply(lambda x: str(x).split('_')[0]) ==
                     df['event_2'].apply(lambda x: str(x).split('_')[0])).astype(int)

print(f'Признаков добавлено: {len(df.columns) - 4}')
print(df[['char_sim','norm_sim','jaccard','len_diff','tok_diff','same_action','same_domain']]
      .describe().round(3).to_string())

**Обоснование нормализации `norm_sim`:**  
Типы `case` включают PascalCase, SCREAMING_SNAKE_CASE, kebab-case и др. — одно и то же событие записано разными форматами.  
После приведения к snake_case (`to_snake`) расстояние между этими парами резко уменьшается, что позволяет строковым моделям их корректно детектировать.  
Разница `norm_sim − char_sim` является диагностическим признаком для типа `case`.

---
## 4. EDA: структура датасета <a id='sec4'></a>

In [ ]:
from IPython.display import Image
Image('figures/fig1_dataset_overview.png')

In [ ]:
# Воспроизводимый код рисунка
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Структура датасета', fontsize=13, fontweight='bold', y=1.02)

ax = axes[0]
tc = df['dup_type'].value_counts().reindex(TYPE_ORDER)
bars = ax.bar(TYPE_ORDER, tc.values, color=[PALETTE[t] for t in TYPE_ORDER], edgecolor='white', width=0.7)
ax.set_xticklabels(TYPE_ORDER, rotation=35, ha='right', fontsize=9)
ax.set_title('Количество пар по типам'); ax.set_ylabel('Количество')
for b,v in zip(bars, tc.values): ax.text(b.get_x()+b.get_width()/2, v+80, f'{v:,}', ha='center', fontsize=8)

ax = axes[1]
lc = df['label'].value_counts()
ax.pie([lc[0],lc[1]], labels=['Не дубль (0)','Дубль (1)'],
       colors=['#bdc3c7','#2ecc71'], autopct='%1.1f%%', startangle=90,
       wedgeprops={'edgecolor':'white','linewidth':2})
ax.set_title('Баланс классов')

ax = axes[2]
pos = df[df.label==1]['dup_type'].value_counts()
pt  = [t for t in TYPE_ORDER if t!='not_duplicate']
ax.barh(pt, [pos.get(t,0) for t in pt], color=[PALETTE[t] for t in pt], edgecolor='white', height=0.65)
ax.set_title('Типы дублей (label=1)'); ax.set_xlabel('Количество пар')
for i,t in enumerate(pt):
    v = pos.get(t,0); ax.text(v+30, i, f'{v:,}  ({v/len(df)*100:.1f}%)', va='center', fontsize=8)

plt.tight_layout(); plt.savefig('figures/fig1_dataset_overview.png', dpi=120, bbox_inches='tight')
plt.show(); print('Рис. 1 готов.')

---
## 5. EDA: символьное сходство <a id='sec5'></a>

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Символьное сходство (char_sim) по типам пар', fontsize=13, fontweight='bold', y=1.02)

ax = axes[0]
bp = ax.boxplot([df[df.dup_type==t]['char_sim'].values for t in TYPE_ORDER],
                patch_artist=True, notch=False,
                medianprops=dict(color='black', linewidth=2.5))
for p,t in zip(bp['boxes'], TYPE_ORDER): p.set_facecolor(PALETTE[t]); p.set_alpha(0.85)
ax.set_xticklabels(TYPE_ORDER, rotation=35, ha='right', fontsize=8)
ax.set_title('Boxplot char_sim по типам'); ax.set_ylabel('char_sim')
ax.axhline(0.75, ls='--', color='#c0392b', lw=1.2, alpha=0.7, label='θ=0.75'); ax.legend(fontsize=8)

ax = axes[1]
ax.hist(df[df.label==1]['char_sim'], bins=50, color='#2ecc71', alpha=0.65, label='Дубли (1)', edgecolor='white')
ax.hist(df[df.label==0]['char_sim'], bins=50, color='#e74c3c', alpha=0.65, label='Не дубли (0)', edgecolor='white')
ax.set_title('char_sim: дубли vs не-дубли'); ax.set_xlabel('char_sim'); ax.legend(fontsize=9)

ax = axes[2]
ms = df.groupby('dup_type')['char_sim'].mean().reindex(TYPE_ORDER)
ss = df.groupby('dup_type')['char_sim'].std().reindex(TYPE_ORDER)
bars = ax.bar(TYPE_ORDER, ms.values, color=[PALETTE[t] for t in TYPE_ORDER],
              yerr=ss.values, capsize=4, edgecolor='white', width=0.7)
ax.set_xticklabels(TYPE_ORDER, rotation=35, ha='right', fontsize=8)
ax.set_title('Среднее char_sim (±std)'); ax.set_ylim(0, 1.3)
for b,v in zip(bars, ms.values): ax.text(b.get_x()+b.get_width()/2, v+0.05, f'{v:.3f}', ha='center', fontsize=8)

plt.tight_layout(); plt.savefig('figures/fig2_char_sim.png', dpi=120, bbox_inches='tight')
plt.show(); print('Рис. 2 готов.')

# Числа
print('\nchar_sim по типам:')
print(df.groupby('dup_type')['char_sim'].agg(['mean','std','min','max']).round(3).to_string())

In [ ]:
# Зона overlap: пары с 0.65 < char_sim < 0.90 — самые трудные для классификации
overlap = df[(df.char_sim > 0.65) & (df.char_sim < 0.90)]
print(f'Пар в overlap-зоне (0.65–0.90): {len(overlap):,} ({len(overlap)/len(df)*100:.1f}%)')
print()
print('Состав overlap-зоны по типам:')
print(overlap.groupby('dup_type').size().sort_values(ascending=False).to_string())
print()
# Примеры overlap из каждого типа
for t in ['semantic','permutation','synonym','not_duplicate','case']:
    sub = overlap[overlap.dup_type==t].head(2)
    if len(sub):
        print(f'\n{t}:')
        for _,r in sub.iterrows():
            print(f'  sim={r.char_sim:.3f}  [{r.label}]  "{r.event_1}"  /  "{r.event_2}"')

---
## 6. EDA: лексические признаки <a id='sec6'></a>

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Лексические признаки датасета', fontsize=13, fontweight='bold', y=1.01)

ax = axes[0,0]
ax.hist(df['len_1'], bins=40, color='#3498db', alpha=0.7, label='event_1', edgecolor='white')
ax.hist(df['len_2'], bins=40, color='#e67e22', alpha=0.7, label='event_2', edgecolor='white')
ax.axvline(df.len_1.mean(), color='#3498db', ls='--', lw=1.5)
ax.axvline(df.len_2.mean(), color='#e67e22', ls='--', lw=1.5)
ax.set_title(f'Длина событий (mean={df.len_1.mean():.0f} симв.)')
ax.set_xlabel('Символов'); ax.legend(fontsize=9)

ax = axes[0,1]
tc1 = df['tok_1'].value_counts().sort_index()
x = tc1.index.tolist()
ax.bar([i-0.2 for i in range(len(x))], tc1.values, 0.4, label='event_1', color='#3498db', edgecolor='white')
tc2 = df['tok_2'].value_counts().sort_index().reindex(x, fill_value=0)
ax.bar([i+0.2 for i in range(len(x))], tc2.values, 0.4, label='event_2', color='#e67e22', edgecolor='white')
ax.set_xticks(range(len(x))); ax.set_xticklabels(x)
ax.set_title(f'Количество токенов (mean={df.tok_1.mean():.1f})')
ax.set_xlabel('Токенов'); ax.legend(fontsize=9)

ax = axes[0,2]
bp = ax.boxplot([df[df.dup_type==t]['len_1'].values for t in TYPE_ORDER], patch_artist=True,
                medianprops=dict(color='black', linewidth=2))
for p,t in zip(bp['boxes'], TYPE_ORDER): p.set_facecolor(PALETTE[t]); p.set_alpha(0.85)
ax.set_xticklabels(TYPE_ORDER, rotation=35, ha='right', fontsize=8)
ax.set_title('Длина event_1 по типам'); ax.set_ylabel('Символов')

ax = axes[1,0]
jm = df.groupby('dup_type')['jaccard'].mean().reindex(TYPE_ORDER)
ax.bar(TYPE_ORDER, jm.values, color=[PALETTE[t] for t in TYPE_ORDER], edgecolor='white', width=0.7)
ax.set_xticklabels(TYPE_ORDER, rotation=35, ha='right', fontsize=8)
ax.set_title('Среднее Jaccard (токены)'); ax.set_ylim(0, 1.15)
for i,v in enumerate(jm.values): ax.text(i, v+0.02, f'{v:.2f}', ha='center', fontsize=8)

ax = axes[1,1]
nm = df.groupby('dup_type')['norm_sim'].mean().reindex(TYPE_ORDER)
ax.bar(TYPE_ORDER, nm.values, color=[PALETTE[t] for t in TYPE_ORDER], edgecolor='white', width=0.7)
ax.set_xticklabels(TYPE_ORDER, rotation=35, ha='right', fontsize=8)
ax.set_title('Среднее norm_sim (после нормализации)'); ax.set_ylim(0, 1.15)
for i,v in enumerate(nm.values): ax.text(i, v+0.02, f'{v:.2f}', ha='center', fontsize=8)

ax = axes[1,2]
ax.hist(df[df.label==1]['len_diff'], bins=30, color='#2ecc71', alpha=0.7, label='Дубли', edgecolor='white')
ax.hist(df[df.label==0]['len_diff'], bins=30, color='#e74c3c', alpha=0.7, label='Не дубли', edgecolor='white')
ax.set_title('Разница длин |len(e1)−len(e2)|'); ax.set_xlabel('Символов'); ax.legend(fontsize=9)

plt.tight_layout(); plt.savefig('figures/fig3_lexical.png', dpi=120, bbox_inches='tight')
plt.show(); print('Рис. 3 готов.')

---
## 7. EDA: корреляции и форматы именования <a id='sec7'></a>

In [ ]:
# Форматы именования в event_1
print('=== Форматы именования (event_1) ===')
fmt_dist = df['fmt_1'].value_counts()
print(fmt_dist.to_string())

print('\n=== Форматы по dup_type ===')
print(df.groupby(['dup_type','fmt_1']).size().unstack(fill_value=0).to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Признаки и форматы именования', fontsize=13, fontweight='bold', y=1.02)

ax = axes[0]
feat_cols = ['char_sim','norm_sim','jaccard','len_diff','tok_diff',
             'len_1','tok_1','same_action','same_domain','label']
corr = df[feat_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            ax=ax, linewidths=0.4, annot_kws={'size':7.5}, vmin=-1, vmax=1,
            cbar_kws={'shrink':0.8})
ax.set_title('Корреляционная матрица признаков')

ax = axes[1]
top_fmts = df['fmt_1'].value_counts().head(5).index
fd = df.groupby(['dup_type','fmt_1']).size().unstack(fill_value=0)
fd = fd[[c for c in top_fmts if c in fd.columns]].reindex(TYPE_ORDER)
fd.plot(kind='bar', ax=ax, width=0.75, edgecolor='white')
ax.set_xticklabels(TYPE_ORDER, rotation=35, ha='right', fontsize=8)
ax.set_title('Форматы event_1 по типам пар'); ax.set_ylabel('Количество')
ax.legend(fontsize=7, title='Формат')

ax = axes[2]
spl = df.sample(4000, random_state=42)
ax.scatter(spl[spl.label==0]['char_sim'], spl[spl.label==0]['norm_sim'],
           alpha=0.18, s=7, color='#e74c3c', label='Не дубль (0)')
ax.scatter(spl[spl.label==1]['char_sim'], spl[spl.label==1]['norm_sim'],
           alpha=0.18, s=7, color='#2ecc71', label='Дубль (1)')
ax.plot([0,1],[0,1],'k--',lw=0.8,alpha=0.4)
ax.set_xlabel('char_sim'); ax.set_ylabel('norm_sim')
ax.set_title('char_sim vs norm_sim (4 000 пар)')
ax.legend(fontsize=8, markerscale=3)

plt.tight_layout(); plt.savefig('figures/fig4_features.png', dpi=120, bbox_inches='tight')
plt.show(); print('Рис. 4 готов.')

In [ ]:
# Корреляция признаков с label
corr_label = df[feat_cols].corr()['label'].drop('label').sort_values()
print('Корреляция признаков с label (Pearson r):')
print(corr_label.round(4).to_string())

---
## 8. Train / Val / Test split <a id='sec8'></a>

In [ ]:
# Стратификация по dup_type для равномерного распределения типов
out_cols = ['event_1','event_2','label','dup_type',
            'char_sim','norm_sim','jaccard',
            'len_1','len_2','tok_1','tok_2',
            'len_diff','tok_diff','same_action','same_domain']

df_tv, df_test   = train_test_split(df, test_size=0.15, random_state=42, stratify=df['dup_type'])
df_train, df_val = train_test_split(df_tv, test_size=0.176, random_state=42, stratify=df_tv['dup_type'])

print('=== Train / Val / Test split ===')
for name, dfs in [('Train (70%)', df_train), ('Val   (15%)', df_val), ('Test  (15%)', df_test)]:
    pos = dfs.label.sum(); neg = len(dfs)-pos
    print(f'{name}: {len(dfs):6,}  | +:{pos:5,} ({pos/len(dfs)*100:.0f}%)  -:{neg:5,}')

# Проверка равномерности по dup_type
print('\nРаспределение dup_type в splits:')
for name, dfs in [('train', df_train), ('val', df_val), ('test', df_test)]:
    tc = dfs['dup_type'].value_counts().reindex(TYPE_ORDER)
    pct = (tc/len(dfs)*100).round(1)
    print(f'  {name}: ' + '  '.join(f'{t}={v:.0f}%' for t,v in pct.items()))

# Сохранение
df[out_cols].to_csv('data/pairs_full_processed.csv', index=False)
df_train[out_cols].to_csv('data/pairs_train.csv', index=False)
df_val[out_cols].to_csv('data/pairs_val.csv', index=False)
df_test[out_cols].to_csv('data/pairs_test.csv', index=False)
print('\nФайлы сохранены в data/')

---
## 9. Выводы <a id='sec9'></a>

### 9.1 Описание датасета

- **50 000 пар** событий из реальной банковской таксономии, 4 колонки: `event_1`, `event_2`, `label`, `dup_type`.
- **Идеально сбалансированный** датасет: 25 000 дублей (50%) и 25 000 не-дублей (50%). Каждый из 6 типов дублей представлен ровно ~4 167 парами (8.33%).
- **0 NaN, 0 дублирующихся пар** — предобработка не требуется, данные чистые.
- **24 292 уникальных события** — высокая степень повторяемости в парах (каждое событие встречается несколько раз в разных контекстах).

### 9.2 Форматы именования

Абсолютное большинство событий (99.97%) записаны в `snake_case`. Исключения — только пары типа `case`, где один элемент представлен в одном из 8 нестандартных форматов: `PascalCase`, `lowerCamelCase`, `SCREAMING_SNAKE_CASE`, `kebab-case`, `Train-Case`, `flatcase`, `UPPERCASE`, `space case`. Нормализация к snake_case (`to_snake`) устраняет это различие.

### 9.3 Характеристики сходства по типам

| Тип | char_sim (mean) | Диагностика |
|-----|-----------------|-------------|
| `exact` | **1.000 ± 0.000** | Тривиальная детекция |
| `typo` | **0.977 ± 0.012** | Высокое сходство — детектируется строковыми методами |
| `synonym` | **0.871 ± 0.053** | Высокое — строковые методы работают |
| `permutation` | **0.785 ± 0.089** | Среднее — строковые методы частично |
| `semantic` | **0.734 ± 0.098** | Среднее — **требует семантики** |
| `case` | **0.642 ± 0.340** | Высокая дисперсия: часть форматов даёт sim→0 (`flatcase`, `UPPERCASE`) |
| `not_duplicate` | **0.323 ± 0.101** | Хорошо отделяется в целом |

### 9.4 Зона перекрытия (overlap)

**24.3% пар** имеют `char_sim ∈ (0.65, 0.90)` — это зона, где строковые методы ошибаются наиболее часто. Основную трудность представляют:
- `semantic` пары с высоким лексическим сходством (общий объектный путь, разные слова)
- `not_duplicate` из одного домена со схожей структурой
- `case` с форматами, близкими к snake_case (пробел вместо `_`)

### 9.5 Признаки и корреляции

- `char_sim` — наиболее коррелированный признак с `label` (r ≈ 0.72)
- `norm_sim` — после нормализации корреляция ещё выше (r ≈ 0.76), особенно для `case`-пар
- `jaccard` (токенное сходство) — r ≈ 0.70, независимый источник сигнала
- `same_action` и `same_domain` — слабые, но ненулевые предикторы
- `len_diff`, `tok_diff` — отрицательно коррелируют с `label` (дубли обычно близки по длине)

### 9.6 Выводы для выбора модели

**Строковые методы (char_sim, norm_sim + порог)** хорошо справятся с `exact`, `typo`, `synonym` (суммарно 33% датасета), но провалятся на `semantic` (~8%) и на `case`-форматах с нулевым перекрытием (`flatcase`, `UPPERCASE`).  

**TF-IDF + ML** добавит паттерны n-граммов и поднимет качество на `permutation` и `case`.  

**SBERT (sentence embeddings)** необходим для семантической группы: `login_success` и `user_authenticated` имеют char_sim ≈ 0.19, но должны быть размечены как дубли.  

**Целевой Pipeline:** нормализация → feature engineering → TF-IDF+SVM baseline → SBERT fine-tuning → сравнение на hard semantic test set.